# 14. Prototipo con M3 oficial

El prototipo carga M3 porque obtuvo el mayor F1 macro de validación en la corrida oficial. Antes de usarlo, verifica el SHA-256 del peso.

## A. Cargar y verificar el modelo seleccionado

- **Qué hace:** lee la selección de 10 y comprueba el peso de M3.
- **Para qué:** impedir que el prototipo utilice el reentrenamiento posterior.
- **Por qué:** el nombre de una carpeta no identifica por sí solo una corrida neuronal.

In [ ]:
%pip install -q "transformers==4.49.0" "pysentimiento==0.7.3" "emoji==2.16.0" pandas openpyxl

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd
import torch
from IPython.display import display
from pysentimiento.preprocessing import preprocess_tweet
from transformers import AutoModelForSequenceClassification, AutoTokenizer

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB')
else:
    RAIZ = Path.cwd()

SELECCION = RAIZ / 'TABLAS/10_entrenamiento_boliviano/seleccion_modelo.json'
MANIFIESTO = RAIZ / 'TABLAS/10_entrenamiento_boliviano/manifest_corrida_oficial.json'
SALIDA = RAIZ / 'TABLAS/14_prototipo'; SALIDA.mkdir(parents=True, exist_ok=True)
seleccion = json.loads(SELECCION.read_text(encoding='utf-8'))
manifiesto = json.loads(MANIFIESTO.read_text(encoding='utf-8'))
if seleccion['modelo_seleccionado'] != 'M3_SMPS_boliviano':
    raise ValueError('La selección oficial esperada es M3_SMPS_boliviano.')
ruta_modelo = RAIZ / seleccion['ruta_modelo']
esperado = manifiesto['modelos']['M3_SMPS_boliviano']['sha256_model_safetensors']

def sha256(ruta):
    h=hashlib.sha256()
    with open(ruta,'rb') as archivo:
        for bloque in iter(lambda: archivo.read(1024*1024),b''): h.update(bloque)
    return h.hexdigest()
if sha256(ruta_modelo/'model.safetensors') != esperado:
    raise ValueError('El peso de M3 no corresponde a la corrida oficial.')

DISPOSITIVO=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mapa={0:'negativa',1:'neutral',2:'positiva'}
tokenizer=AutoTokenizer.from_pretrained(str(ruta_modelo),local_files_only=True)
modelo=AutoModelForSequenceClassification.from_pretrained(str(ruta_modelo),local_files_only=True).to(DISPOSITIVO).eval()
print('Modelo:', seleccion['modelo_seleccionado']); print('SHA-256 verificado:', esperado)

## B. Clasificar con el protocolo oficial

- **Qué hace:** divide textos largos con máximo 128 piezas y solapamiento 32.
- **Para qué:** conservar el procedimiento utilizado en la evaluación oficial.
- **Por qué:** la polaridad final se obtiene promediando las probabilidades Softmax de todos los tramos.

In [ ]:
def predecir(texto, tok=tokenizer, mod=modelo):
    entrada=preprocess_tweet(str(texto),lang='es')
    lote=tok(entrada,max_length=128,truncation=True,stride=32,
             return_overflowing_tokens=True,padding=True,return_tensors='pt')
    lote.pop('overflow_to_sample_mapping',None)
    lote={k:v.to(DISPOSITIVO) for k,v in lote.items()}
    with torch.inference_mode():
        probabilidades=torch.softmax(mod(**lote).logits,dim=-1).mean(dim=0).cpu().numpy()
    return {'polaridad':mapa[int(probabilidades.argmax())],
            'negativa':float(probabilidades[0]),'neutral':float(probabilidades[1]),
            'positiva':float(probabilidades[2]),'tramos':int(lote['input_ids'].shape[0])}

## C. Probar cinco textos sintéticos

- **Qué hace:** clasifica ejemplos creados para la demostración.
- **Para qué:** mostrar la salida sin revelar publicaciones del corpus.
- **Por qué:** una demostración técnica no requiere exponer datos privados.

In [ ]:
EJEMPLOS=['Hoy aprobé la materia y estoy bien feliz, valió la pena estudiar.',
          'Qué macana, perdí el examen y me siento muy frustrado.',
          'Mañana nomás hay clases normales desde las ocho.',
          'No me fue bien, pero voy a seguir intentando hasta mejorar.',
          'Qué buena onda, terminamos el proyecto entre todos.']
tabla=pd.DataFrame([{'texto':texto,**predecir(texto)} for texto in EJEMPLOS])
tabla.to_excel(SALIDA/'resultados_ejemplos.xlsx',index=False)
display(tabla.style.format({'negativa':'{:.2%}','neutral':'{:.2%}','positiva':'{:.2%}'}))
display(tabla.head(10))

## D. Interfaz opcional

- **Qué hace:** muestra las tres probabilidades en Gradio.
- **Para qué:** facilitar una demostración visual.
- **Por qué:** la tabla anterior funciona aunque Gradio no esté instalado.

In [ ]:
EJECUTAR_GRADIO=False
if EJECUTAR_GRADIO:
    try:
        import gradio as gr
        def analizar(texto):
            r=predecir(texto); return {k:r[k] for k in ['negativa','neutral','positiva']}
        gr.Interface(fn=analizar,inputs=gr.Textbox(lines=5,label='Texto'),
                     outputs=gr.Label(num_top_classes=3,label='Polaridad y probabilidades'),
                     title='Clasificación de polaridad con M3 oficial').launch(share=False)
    except ImportError:
        print('Gradio no está instalado; utiliza la tabla de la sección C.')
else:
    print('Cambia EJECUTAR_GRADIO a True para abrir la interfaz opcional.')